## 7.2 Simple RNN实现 - 模型训练

#### 1、这一节我们要学什么？

##### 1.1 前一节我们已经完成了模型搭建
我们已经知道了如何用 PyTorch 搭建一个最基础的 Simple RNN 模型：

输入序列 $\rightarrow$ `nn.RNN` $\rightarrow$ 取最后时间步隐藏状态 $\rightarrow$ `Linear` $\rightarrow$ 输出类别分数

也就是说，现在我们已经有了**模型结构**。🧠

##### 1.2 但是只有模型结构还不能真正训练
一个完整的深度学习训练流程，至少还需要补齐下面这些部分：

- 定义损失函数 `loss function`
- 定义优化器 `optimizer`
- 定义学习率调度器 `scheduler`
- 定义训练函数 `train_one_epoch()`
- 定义验证函数 `validate_one_epoch()`
- 编写主训练循环 `for epoch in ...`
- 在训练结束后进行模型验证与效果观察

##### 1.3 这一节的目标
这一节我们重点不是研究复杂案例，而是先把 **RNN 的标准训练模板** 完整打通。✅

学完这一节之后，应该能够清楚理解：

- 一个 RNN 分类任务训练时，代码整体结构是什么
- 每一步为什么这样写
- 每一部分张量在训练中是怎么流动的
- 为什么要区分训练模式和验证模式
- `scheduler` 在哪里更新最合理


#### 二、先明确：RNN 训练流程和普通神经网络本质上是一样的

##### 2.1 训练流程的核心并没有因为 RNN 而改变
虽然 RNN 处理的是序列数据，但从训练框架上看，它和我们前面学过的 MLP、CNN 是一样的。

完整流程仍然是：

输入数据 $\rightarrow$ 前向传播 $\rightarrow$ 计算损失 $\rightarrow$ 反向传播 $\rightarrow$ 参数更新

也就是：

- `model(x)` 得到预测结果
- `criterion(outputs, labels)` 计算损失
- `loss.backward()` 反向传播
- `optimizer.step()` 更新参数

##### 2.2 RNN 和 MLP / CNN 的主要区别在哪？
区别不在训练框架，而在于：

- 输入数据形状不同
- 模型内部的计算结构不同

例如：

- MLP 常见输入：  
  $[batch\_size, features]$
- CNN 常见输入：  
  $[batch\_size, channels, height, width]$
- RNN 常见输入（`batch_first=True`）：  
  $[batch\_size, seq\_len, input\_size]$

RNN 的训练逻辑不难，真正容易出错的是**输入维度**。⚠️

#### 三、我们先明确当前任务类型

##### 3.1 我们这里先学习最基础的 many-to-one 分类任务
也就是：

一整条序列 $\rightarrow$ 输出一个类别

例如：

- 一句话判断情感类别
- 一段时间序列判断涨跌趋势
- 一组传感器序列判断设备状态

因此模型输出的是：

$[batch\_size, num\_classes]$

其中：

- `batch_size` 维度表示对应的一个样本
- `num_classes` 维度表示对应样本所有类别的原始分数 `logits`

##### 3.2 为什么输出不是每个时间步一个类别？
因为我们现在的模型中：

```python
last_output = output[:, -1, :]
out = self.fc(last_output)
``` 
我们只取了最后一个时间步的隐藏状态，再送入全连接层。

所以这个模型表达的是：
>整条序列最终只输出一个分类结果
这正是 many-to-one 任务的典型结构。

#### 4. 训练一个 RNN 模型需要准备哪些组件？
##### 4.1 一共需要 6 个核心部分

训练一个基础的 RNN 分类模型，一般至少包括：
* 数据加载器 DataLoader
* 模型 model
* 损失函数 criterion
* 优化器 optimizer
* 学习率调度器 scheduler
* 训练与验证函数

##### 4.2 它们各自负责什么？
* DataLoader：按 batch 提供数据
* model：定义网络结构
* criterion：衡量预测与真实标签之间的误差
* optimizer：根据梯度更新参数
* scheduler：动态调整学习率
* train/validate：封装训练和验证流程

##### 4.3 为什么要把训练和验证写成函数？
因为这样做更规范，也更方便复用。✅

例如：
* 每个 epoch 都要重复训练
* 每个 epoch 结束后都要验证
* 后面换模型时，这套框架基本还能直接复用

所以这是一种很标准的工程写法。

#### 5、准备 device ✅

##### 5.1 为什么要有 device？
因为模型和数据需要放到同一个设备上运行。

常见设备有：

- CPU
- GPU

所以通常会写：

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
```

##### 5.2 模型要放到 device 上
例如：

```python
model = SimpleRNNModel(...).to(device)
```

##### 5.3 每个 batch 的数据也要放到 device 上
```python
x = x.to(device)
labels = labels.to(device)
```

##### 5.4 为什么模型和数据必须在同一个设备？
否则就会报错，例如：

```python
Expected all tensors to be on the same device
```

所以这个步骤虽然简单，但非常重要。⚠️


#### 6、定义损失函数 ✅

##### 6.1 先看当前任务是什么
如果是二分类任务，与 MLP 和 RNN 一样，通常选择：

```python
nn.BCEWithLogitsLoss()
```

我们现在是多分类任务，因此损失函数一般选择：

```python
nn.CrossEntropyLoss()
```

⚠️ 注意：

- 二分类任务的损失函数会自动在输出层应用激活函数 $Sigmoid$
- 多分类任务的损失函数会自动在输出层应用激活函数 $Softmax$
- 并且多分类任务中，标签通常使用**类别索引**，不需要手动写 One-Hot

##### 6.2 为什么这里用交叉熵？
因为模型最后输出的形状是：

$[batch\_size, num\_classes]$

表示每个样本对应每个类别的“原始分数” $logits$

而 `CrossEntropyLoss` 正好适用于：

- 多分类任务
- 模型输出未经过 $Softmax$ 的 $logits$
- 标签是类别索引而不是 one-hot

##### 6.3 这里不需要手动加 Softmax
这一点非常重要。⭐

与 MLP 和 CNN 一样，如果你使用的是：

```python
criterion = nn.CrossEntropyLoss()
```

那么模型最后一层只需要输出：

```python
out = self.fc(last_output)
```

不要手动写：

```python
out = torch.softmax(self.fc(last_output), dim=1)
```

因为 `CrossEntropyLoss` 内部已经自动完成了：

- `LogSoftmax`
- `NLLLoss`

也就是说，它本质上等价于：

```python
log_probs = torch.log_softmax(outputs, dim=1)
loss = nll_loss(log_probs, labels)
```

所以在模型里再手动加一层 $Softmax$，通常是多余的，反而可能影响数值稳定性。

##### 6.4 标签是什么形式？
例如多分类时：

```python
labels = tensor([0, 2, 1, 0])
```

也就是**类别索引形式**，而不是 One-Hot：

```python
[[1,0,0], [0,0,1], [0,1,0], [1,0,0]]
```

这里需要特别注意：

`CrossEntropyLoss` **并不是简单地“先把标签显式转换成 one-hot 再计算”**，而是直接根据类别索引去取对应类别的位置来计算交叉熵。

所以在实际使用中，你只需要提供：

- 模型输出：$[batch\_size, num\_classes]$
- 标签：$[batch\_size]$，其中每个值是类别编号

例如：

```python
outputs = torch.tensor([
    [2.1, 0.3, -1.2],
    [0.1, 1.8, 0.5]
])

labels = torch.tensor([0, 1])
```

表示：

- 第一个样本的真实类别是第 $0$ 类
- 第二个样本的真实类别是第 $1$ 类

然后可以直接计算：

```python
criterion = nn.CrossEntropyLoss()
loss = criterion(outputs, labels)
```

##### 6.5 小结
这一部分你要记住 4 个核心点：

1. 二分类常用：

```python
nn.BCEWithLogitsLoss()
```

2. 多分类常用：

```python
nn.CrossEntropyLoss()
```

3. 使用 `CrossEntropyLoss()` 时，模型输出应当是**原始 logits**，不要手动加 $Softmax$

4. 标签使用**类别索引**即可，不需要手动写 One-Hot

#### 7、定义优化器 ✅

##### 7.1 最常用的是 Adam
在学习阶段，我们通常优先使用：

```python
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
```

##### 7.2 Adam 的优点
它是一个非常常用的自适应优化器，优点包括：

- 学习率调整更灵活
- 收敛通常比普通 SGD 更快
- 对初学者更友好
- 对很多任务效果都比较稳定

#### 8、定义 scheduler ✅

##### 8.1 学习率不是一成不变的
训练一开始，较大的学习率有助于快速靠近较优区域。

但到了训练后期，如果学习率一直不变，可能会出现：

- 在最优点附近来回震荡
- 难以进一步细调参数
- 验证集效果提升缓慢

所以我们经常会在训练过程中逐步降低学习率。📉

##### 8.2 scheduler 的作用
按照某种规则自动调整优化器中的学习率。

##### 8.3 最常见写法
例如可以使用 `ReduceLROnPlateau`：

```python
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=10,
    verbose=True
)
```

它表示：

- 每训练 $10$ 个 epoch（`patience`）
- 如果 validation loss 没有下降（`mode='min'`）
- 将学习率乘以 $0.1$（`factor`）

#### 9、定义训练函数 `train_one_epoch()` ✅

##### 9.1 训练函数的职责
训练函数通常负责：

- 切换到训练模式
- 遍历训练集
- 前向传播
- 计算损失
- 清空旧梯度
- 反向传播
- 参数更新
- 统计平均损失与准确率

##### 9.2 标准结构如下
```python
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    for x, labels in dataloader:
        x = x.to(device)
        labels = labels.to(device)
        
        outputs = model(x)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * x.size(0)
        
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc
```

##### 9.3 要点 💡

##### 9.3.1 `model.train()`
把模型切换到训练模式。

因为有些层在训练和验证时行为不同，例如：

- `Dropout`
- `BatchNorm`

虽然我们当前这个 `Simple RNN` 模型里还没有这些层，但养成这个习惯非常重要。✅

##### 9.3.2 `for x, labels in dataloader:`
表示按 batch 遍历训练数据。

假设：

- `x.shape = [32, 10, 8]`
- `labels.shape = [32]`

那么表示：

- 每次取 $32$ 个样本
- 每个样本有 $10$ 个时间步
- 每个时间步有 $8$ 个特征

##### 9.3.3 `outputs = model(x)`
把当前 batch 输入模型，得到预测结果。

由于我们当前做的是分类任务，所以输出形状通常是：

$[batch\_size, num\_classes]$

表示 $32$ 个样本，每个样本输出 $3$ 个类别分数。

##### 9.3.4 `loss = criterion(outputs, labels)`
计算当前 batch 的损失。

- `outputs` 是预测 `logits`
- `labels` 是真实类别索引

##### 9.3.5 `optimizer.zero_grad()`
清空上一轮累积的梯度。

PyTorch 中梯度默认是累积的，因此每次反向传播之前都必须先清零。

##### 9.3.6 `loss.backward()`
根据当前损失自动计算所有参数的梯度。

对于 RNN 来说，这里面内部完成的正是我们前面学过的 BPTT 相关梯度传播。🧠

##### 9.3.7 `optimizer.step()`
根据已经计算好的梯度，更新模型参数。

##### 9.3.8 `loss.item()` 为什么乘以 `x.size(0)`？
因为 `loss.item()` 通常是当前 batch 的平均损失。

如果不同 batch 的样本数可能不完全一致，那么为了最后精确计算整个 epoch 的平均损失，通常写成：

```python
running_loss += loss.item() * x.size(0)
```

最后再除以总样本数：

```python
epoch_loss = running_loss / total
```

##### 9.3.9 `outputs.argmax(dim=1)` 是什么？
在类别维度上取分数最大的那个类别编号。

例如：

```python
outputs = [
    [1.2, 0.3, -0.5],
    [0.1, 2.4, 1.8]
]
```

那么：

```python
preds = [0, 1]
```

#### 10、验证函数 `validate_one_epoch()` ✅

##### 10.1 验证函数的职责
验证函数通常负责：

- 切换到验证模式
- 禁止梯度计算
- 遍历验证集
- 前向传播
- 计算损失
- 统计准确率

它和训练函数很像，但有两个关键区别：

- 不做反向传播
- 不更新参数

##### 10.2 标准结构如下
```python
def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x, labels in dataloader:
            x = x.to(device)
            labels = labels.to(device)
            
            outputs = model(x)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * x.size(0)
            
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc
```

##### 10.3 要点

##### 10.3.1 `model.eval()` 的作用
把模型切换到验证 / 测试模式。

这样一些层会采用验证时的行为，而不是训练时的行为。

##### 10.3.2 `torch.no_grad()` 的作用
验证阶段不需要计算梯度，因此可以：

- 节省显存 / 内存
- 加快验证速度

##### 10.3.3 为什么验证时不能更新参数？
因为验证集的作用是：

模拟模型在“未见过的数据”上的表现。

如果你在验证阶段也更新参数，那验证结果就不客观了。

#### 11、定义主训练循环 ✅

##### 11.1 主训练循环的职责
主训练循环负责把前面的组件串起来：

- 循环多个 epoch
- 每个 epoch 先训练，再验证
- 更新 scheduler
- 打印训练日志
- 保存最佳模型（可选）

##### 11.2 一个标准写法如下
```python
num_epochs = 20

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    val_loss, val_acc = validate_one_epoch(
        model, val_loader, criterion, device
    )
    
    scheduler.step()
    
    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {val_loss:.4f}, Val   Acc: {val_acc:.4f}")
    print("-" * 50)
```

##### 11.3 要点

##### 11.3.1 `scheduler` 应该放在哪里更新？
对于这种按 epoch 更新的 `scheduler`，最常见的写法就是：

```python
scheduler.step()
```

放在每个 epoch 结束后。

##### 11.3.2 为什么不放在每个 batch 后？
因为设计就是：

按 epoch 作为单位衰减学习率。

所以通常一个 epoch 结束后更新一次最合理。

##### 11.3.3 ⚠️ 注意：不同 scheduler 更新位置可能不同
- `StepLR`：通常每个 epoch 更新
- `ReduceLROnPlateau`：通常根据验证集 loss 更新
- 某些 warmup / cosine 策略：可能每个 batch 更新

#### 12、模型最终验证 ✅

##### 12.1 最基础的方式
训练完成后，我们至少应该在验证集或测试集上再跑一遍评估。

例如：

```python
test_loss, test_acc = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")
```

##### 12.2 为什么测试也可以复用验证函数？
因为测试阶段和验证阶段本质上都一样：

- 不更新参数
- 不计算梯度
- 只做前向传播和评估

所以很多时候验证函数也可以直接作为测试函数使用。

##### 12.3 更完整的评估还可以包括什么？
后面你还可以进一步加上：

- `confusion matrix`
- `precision`
- `recall`
- `F1-score`

#### 13、完整模型构建 + 训练代码

In [ ]:
import torch
import torch.nn as nn

# device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 构建模型
class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super().__init__()
        # RNN层
        self.rnn = nn.RNN(
            input_size=input_size, 
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )
        # 全连接层
        self.fc = nn.Linear(
            in_features=hidden_size,
            out_features=output_size
        )

    def forward(self, x):
        # RNN层
        out, hn = self.rnn(x)
        # 取最后一个时间步的输出
        last_h = out[:, -1, :]
        # 全连接层
        output = self.fc(last_h)
        return output

# 实例化模型
model = SimpleRNN(input_size=10, hidden_size=20, output_size=5)
model.to(device)

# 定义损失函数
criterion = nn.CrossEntropyLoss()

# 定义优化器
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# 定义 scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=5
)

# 定义训练方法
def train_one_epoch(model, dataloader, criterion, optimizer, divice):
    model.train()
    total_loss = 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    
    return epoch_loss, epoch_acc

# 定义评估方法
def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

# 定义训练主循环
# 由于现在没有实际的数据加载器，所以这里忽略主循环的实现细节
